# Chennai Urban Arterial Traffic Forecasting Pipeline
### End-to-End Autoregressive Machine Learning Pipeline with Calibrated Benchmark Data

**Platform:** Chennai Traffic Intelligence & Multidimensional Visualization Platform  
**Models:** HistGradientBoosting (GBDT) & Random Forest  
**Forecast Horizons:** +15m, +30m (Primary), +45m, +60m  
**Data Classification:** `SIMULATED DEVELOPMENT BENCHMARK` (Scientific Honesty Notice: Synthetic multi-day dataset calibrated to Chennai road network geometry and diurnal cycles for pipeline validation)

---

## 1. Environment Setup & Dependency Imports

In [ ]:
import os
import sys
import json
import math
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
import joblib

print(f"Python Version: {sys.version}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version:  {np.__version__}")

## 2. Configuration & Constants

In [ ]:
RANDOM_SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

PRIMARY_TARGET = "congestion_index"
DEFAULT_HORIZON = 30

GATE_MIN_UNIQUE_DAYS = 7
GATE_MIN_RECORDS = 1000
GATE_MIN_OBSERVATIONS_PER_ROAD = 50

## 3. Calibrated Multi-Day Benchmark Dataset Generation (14 Days)

In [ ]:
# Define Chennai monitored road network specifications
CORRIDORS = [
    {"road_id": "ROAD_ANNA_SALAI_1", "road_name": "Anna Salai (Central-Gemini)", "speed_limit": 50.0, "capacity": 4200, "lanes": 4},
    {"road_id": "ROAD_ANNA_SALAI_2", "road_name": "Anna Salai (Gemini-Guindy)", "speed_limit": 50.0, "capacity": 4000, "lanes": 4},
    {"road_id": "ROAD_GST_1", "road_name": "GST Road (Guindy-Airport)", "speed_limit": 60.0, "capacity": 4800, "lanes": 4},
    {"road_id": "ROAD_GST_2", "road_name": "GST Road (Airport-Tambaram)", "speed_limit": 60.0, "capacity": 4600, "lanes": 4},
    {"road_id": "ROAD_OMR_1", "road_name": "OMR IT Corridor (Madhya Kailash-SRP)", "speed_limit": 60.0, "capacity": 4500, "lanes": 3},
    {"road_id": "ROAD_OMR_2", "road_name": "OMR IT Corridor (SRP-Sholinganallur)", "speed_limit": 60.0, "capacity": 4200, "lanes": 3},
    {"road_id": "ROAD_PH_1", "road_name": "Poonamallee High Rd (Central-Kilmauk)", "speed_limit": 50.0, "capacity": 3800, "lanes": 3},
    {"road_id": "ROAD_PH_2", "road_name": "Poonamallee High Rd (Kilmauk-Koyambedu)", "speed_limit": 50.0, "capacity": 3600, "lanes": 3},
    {"road_id": "ROAD_100FT_1", "road_name": "100 Feet Rd / Jawaharlal Nehru Salai", "speed_limit": 50.0, "capacity": 4000, "lanes": 3},
    {"road_id": "ROAD_ECR_1", "road_name": "East Coast Road (Thiruvanmiyur-Akkarai)", "speed_limit": 60.0, "capacity": 3200, "lanes": 2},
    {"road_id": "ROAD_MOUNT_POON_1", "road_name": "Mount-Poonamallee Rd (Guindy-Porur)", "speed_limit": 50.0, "capacity": 3500, "lanes": 3},
    {"road_id": "ROAD_ARCOT_1", "road_name": "Arcot Road (Kodambakkam-Porur)", "speed_limit": 40.0, "capacity": 2800, "lanes": 2},
    {"road_id": "ROAD_SP_ROAD_1", "road_name": "Sardar Patel Road (Guindy-Adyar)", "speed_limit": 50.0, "capacity": 3600, "lanes": 3},
    {"road_id": "ROAD_MARINA_1", "road_name": "Kamarajar Salai / Marina Beach Road", "speed_limit": 50.0, "capacity": 3400, "lanes": 3}
]

print(f"Defined {len(CORRIDORS)} Chennai arterial corridors.")

In [ ]:
def generate_benchmark_records(num_days=14, seed=42):
    random.seed(seed)
    start_date = datetime(2026, 8, 17, 0, 0)
    records = []
    
    for day_idx in range(num_days):
        curr_date = start_date + timedelta(days=day_idx)
        date_str = curr_date.strftime("%Y-%m-%d")
        is_weekend = curr_date.weekday() >= 5
        
        for hour in range(24):
            is_morning_peak = (8 <= hour <= 11) and not is_weekend
            is_evening_peak = (17 <= hour <= 20) and not is_weekend
            is_peak = is_morning_peak or is_evening_peak
            rain = 8.5 if (day_idx in [4, 5, 11] and 8 <= hour <= 12) else 0.0
            
            for c in CORRIDORS:
                cap, limit, rid = c["capacity"], c["speed_limit"], c["road_id"]
                vol_factor = 0.90 if is_peak else (0.35 if is_weekend else 0.50)
                vol = int(cap * vol_factor * (1.0 + 0.05 * random.uniform(-1, 1)))
                util = vol / cap
                
                spd_factor = max(0.18, 0.85 - (util * 0.6) - (0.15 if rain > 0 else 0.0))
                spd = round(limit * spd_factor * (1.0 + 0.03 * random.uniform(-1, 1)), 1)
                spd = max(5.0, min(limit, spd))
                
                speed_deficit = max(0.0, (limit - spd) / limit)
                ci = round((0.45 * min(1.0, util / 1.2) * 100.0) + (0.55 * speed_deficit * 100.0), 1)
                ci = max(0.0, min(100.0, ci))
                
                records.append({
                    "road_id": rid,
                    "timestamp": f"{date_str}T{hour:02d}:00:00+05:30",
                    "date": date_str,
                    "hour": hour,
                    "is_weekend": int(is_weekend),
                    "is_peak_hour": int(is_peak),
                    "vehicle_count": vol,
                    "average_speed": spd,
                    "road_capacity": cap,
                    "speed_limit": limit,
                    "lane_count": c["lanes"],
                    "rainfall": rain,
                    "congestion_index": ci
                })
    return pd.DataFrame(records)

df_raw = generate_benchmark_records(14)
print(f"Generated benchmark dataset: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns.")

## 4. Data Sufficiency Gate Audit

In [ ]:
unique_days = df_raw["date"].nunique()
total_rows = len(df_raw)
min_obs_per_road = df_raw.groupby("road_id").size().min()

print(f"Unique Days: {unique_days} (Required: >= {GATE_MIN_UNIQUE_DAYS})")
print(f"Total Records: {total_rows} (Required: >= {GATE_MIN_RECORDS})")
print(f"Min Observations/Road: {min_obs_per_road} (Required: >= {GATE_MIN_OBSERVATIONS_PER_ROAD})")

assert unique_days >= GATE_MIN_UNIQUE_DAYS, "Gate Failed: Insufficient Days"
assert total_rows >= GATE_MIN_RECORDS, "Gate Failed: Insufficient Records"
print("\n[GATE PASSED] Dataset satisfied all sufficiency criteria for ML training.")

## 5. Feature Engineering (Strict Zero Future Leakage)

In [ ]:
df_clean = df_raw.sort_values(["road_id", "timestamp"]).reset_index(drop=True)

# 1. Cyclical time
df_clean["hour_sin"] = np.sin(2 * np.pi * df_clean["hour"] / 24.0)
df_clean["hour_cos"] = np.cos(2 * np.pi * df_clean["hour"] / 24.0)

# 2. Strict backward lags (T-1, T-2, T-3)
grouped = df_clean.groupby("road_id")
df_clean["vehicle_count_lag_1"] = grouped["vehicle_count"].shift(1)
df_clean["vehicle_count_lag_2"] = grouped["vehicle_count"].shift(2)
df_clean["vehicle_count_lag_3"] = grouped["vehicle_count"].shift(3)

df_clean["speed_lag_1"] = grouped["average_speed"].shift(1)
df_clean["speed_lag_2"] = grouped["average_speed"].shift(2)
df_clean["speed_lag_3"] = grouped["average_speed"].shift(3)

df_clean["congestion_lag_1"] = grouped["congestion_index"].shift(1)
df_clean["congestion_lag_2"] = grouped["congestion_index"].shift(2)

# 3. Rolling statistics on shift(1) - strictly excluding observation at T
df_clean["rolling_speed_mean_3h"] = grouped["average_speed"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df_clean["rolling_vol_mean_3h"] = grouped["vehicle_count"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df_clean["rolling_ci_mean_3h"] = grouped["congestion_index"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

# 4. Target variable: Congestion Index at T+1 (+30m lead)
df_clean["target_congestion_index"] = grouped["congestion_index"].shift(-1)

# Drop cold start rows and end-of-series rows
df_supervised = df_clean.dropna().reset_index(drop=True)
print(f"Supervised dataset prepared: {len(df_supervised)} rows.")

## 6. Chronological Train / Validation / Test Splitting

In [ ]:
df_sorted = df_supervised.sort_values("timestamp").reset_index(drop=True)
n = len(df_sorted)

train_df = df_sorted.iloc[:int(n * TRAIN_RATIO)]
val_df = df_sorted.iloc[int(n * TRAIN_RATIO):int(n * (TRAIN_RATIO + VAL_RATIO))]
test_df = df_sorted.iloc[int(n * (TRAIN_RATIO + VAL_RATIO)):]

FEATURE_COLS = [
    "hour_sin", "hour_cos", "is_weekend", "is_peak_hour",
    "road_capacity", "speed_limit", "lane_count",
    "vehicle_count_lag_1", "vehicle_count_lag_2", "vehicle_count_lag_3",
    "speed_lag_1", "speed_lag_2", "speed_lag_3",
    "congestion_lag_1", "congestion_lag_2",
    "rolling_speed_mean_3h", "rolling_vol_mean_3h", "rolling_ci_mean_3h",
    "rainfall"
]

X_train, y_train = train_df[FEATURE_COLS], train_df["target_congestion_index"].values
X_val, y_val = val_df[FEATURE_COLS], val_df["target_congestion_index"].values
X_test, y_test = test_df[FEATURE_COLS], test_df["target_congestion_index"].values

print(f"Train set: {len(X_train)} | Val set: {len(X_val)} | Test set: {len(X_test)}")

## 7. Baseline Benchmarks Evaluation

In [ ]:
def eval_metrics(y_true, y_pred):
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = float(1.0 - (np.sum((y_true - y_pred) ** 2) / ss_tot)) if ss_tot > 0 else 0.0
    return {"mae": round(mae, 3), "rmse": round(rmse, 3), "r2": round(r2, 4)}

# Baseline 1: Persistence (y_pred = congestion_lag_1)
pers_val_preds = X_val["congestion_lag_1"].values
pers_test_preds = X_test["congestion_lag_1"].values
pers_val = eval_metrics(y_val, pers_val_preds)
pers_test = eval_metrics(y_test, pers_test_preds)

print(f"Persistence Baseline -> Val MAE: {pers_val['mae']} | Test MAE: {pers_test['mae']}")

## 8. Candidate Model Training & Selection

In [ ]:
# Candidate A: Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train, y_train)
rf_val_preds = rf.predict(X_val)
rf_val = eval_metrics(y_val, rf_val_preds)

# Candidate B: HistGradientBoosting
gb = HistGradientBoostingRegressor(max_iter=150, max_depth=8, learning_rate=0.08, random_state=RANDOM_SEED)
gb.fit(X_train, y_train)
gb_val_preds = gb.predict(X_val)
gb_val = eval_metrics(y_val, gb_val_preds)

print(f"Random Forest       -> Val MAE: {rf_val['mae']} | RMSE: {rf_val['rmse']} | R2: {rf_val['r2']}")
print(f"Gradient Boosting   -> Val MAE: {gb_val['mae']} | RMSE: {gb_val['rmse']} | R2: {gb_val['r2']}")

best_model = gb if gb_val["mae"] <= rf_val["mae"] else rf
model_name = "HistGradientBoostingRegressor" if best_model == gb else "RandomForestRegressor"
print(f"\nSELECTED WINNING MODEL: {model_name}")

## 9. Final Test Set Evaluation & Baseline Comparison

In [ ]:
final_test_preds = best_model.predict(X_test)
final_test_metrics = eval_metrics(y_test, final_test_preds)

improvement = ((pers_test["mae"] - final_test_metrics["mae"]) / pers_test["mae"]) * 100.0

print("=" * 60)
print("FINAL HELD-OUT TEST PERFORMANCE:")
print(f"  Persistence Baseline MAE: {pers_test['mae']}")
print(f"  Winning Model MAE:        {final_test_metrics['mae']}")
print(f"  Winning Model RMSE:       {final_test_metrics['rmse']}")
print(f"  Winning Model R-squared:  {final_test_metrics['r2']}")
print(f"  Outperformance:           {improvement:+.1f}% MAE improvement")
print("=" * 60)